# Deep Rotated MNIST: first versus last layers

Load the checkpoint specified below without retraining. Compare the first and last **post-ReLU hidden layers** within the shared backbone and each task branch: layers 0 and 4, rather than the prediction outputs. This compares learned hidden features of equal width instead of mixing the 10-digit, 2-parity, and 1-angle output spaces.

The code currently selects `deep-25-epoch-model-new.pkl`. Its training history determines how to interpret the results: a checkpoint trained before the rotation-loss corrections retains those limitations. Similarity alone does not establish successful rotation prediction.


In [ ]:
from pathlib import Path
import random
import sys

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

# Run from either the repository root or the rotated-MNIST directory.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "model.py").exists():
    PROJECT_DIR = PROJECT_DIR / "rotated-MNIST"
if not (PROJECT_DIR / "model.py").exists():
    raise FileNotFoundError("Run from the repository root or rotated-MNIST directory")
sys.path.insert(0, str(PROJECT_DIR))
from model import MultiTaskDNN
from data_setup import create_dataloaders

SEED = 67
N_SAMPLES = 1000  # RSA uses N * (N - 1) / 2 pairwise distances.
BATCH_SIZE = 128
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


## Load the saved deep model

The state dictionary matches five shared blocks and five hidden blocks per task, all width 16. Inference runs on CPU in evaluation mode, using stored batch-normalization statistics. No optimizer or training loop is created.


In [ ]:
model_cfg = {
    "input_shape": 784,
    "shared_layers": 5,
    "task_layers": 5,
    "shared_nodes": 16,
    "task_nodes": 16,
    "digit_out": 10,
    "parity_out": 2,
    "rotation_out": 1,
}
checkpoint_path = PROJECT_DIR / "deep-25-epoch-model-new.pkl"
model = MultiTaskDNN(model_cfg, {"activation": "relu"})
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()
print(f"Loaded {checkpoint_path.name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


## Collect matched activations: the unit of comparison

For each branch, let $X,Y\in\mathbb{R}^{n\times d}$ contain first- and last-layer activations, with $n=1000$ examples and $d=16$ neurons. Row $i$ must refer to the **same rotated image** in both matrices. Neuron columns need not have matching meanings; the metrics compare representations without requiring neuron-by-neuron correspondence.

Use the first 1,000 held-out MNIST examples in dataset order. This is a reproducible convenience sample, not a balanced or randomly selected subset; the displayed digit counts help reveal imbalance. Every image receives a random angle sampled uniformly over $[0,2\pi]$ by the data loader. Angles and labels are metadata, not inputs to the similarity formulas.

Collect activations once and reuse them for every metric. Otherwise fresh rotations would change the stimuli between comparisons. Reset the seed immediately before collection; reproducibility is within the same software environment. Evaluation mode fixes batch-normalization statistics, so the representation of one image does not depend on its batch companions. Disabling gradients saves memory without changing inference values. Batch size 128 controls computational cost, not the definition of a sample.

RSA needs $n(n-1)/2=499{,}500$ unique pairs at $n=1000$, so this sample size limits quadratic memory and runtime. Cast to float64 before matrix products to reduce numerical error; this does not recover precision already lost during float32 model inference.


In [ ]:
data_cfg = {
    "dataset_path": str(PROJECT_DIR / "DNN_Data"),
    "batch_size": BATCH_SIZE,
    "num_workers": 0,
    "max_digit": 9,
    "max_rotation_degrees": 360,
}
_, test_loader = create_dataloaders(data_cfg)
comparisons = {
    branch: (f"{branch}_Layer_0", f"{branch}_Layer_4")
    for branch in ["Shared", "Digit", "Parity", "Rotation"]
}
layer_names = [name for pair in comparisons.values() for name in pair]
activation_batches = {name: [] for name in layer_names}
digit_batches, angle_batches = [], []
collected = 0
random.seed(SEED)
torch.manual_seed(SEED)
model.eval()
with torch.no_grad():
    for images, digits, _, angle_deg in test_loader:
        take = min(N_SAMPLES - collected, len(images))
        _, _, _, acts = model(images[:take])
        for name in layer_names:
            activation_batches[name].append(acts[name].cpu())
        digit_batches.append(digits[:take])
        angle_batches.append(angle_deg[:take])
        collected += take
        if collected >= N_SAMPLES:
            break

activations = {
    name: torch.cat(batches).numpy().astype(np.float64)
    for name, batches in activation_batches.items()
}
digits = torch.cat(digit_batches).numpy()
angles_deg = torch.cat(angle_batches).numpy()
assert all(a.shape == (collected, 16) for a in activations.values())
print(f"Collected {collected} matched test examples across {len(activations)} layers")
display(pd.DataFrame({"digit": digits}).value_counts().sort_index())


## Mathematical definitions and choices

### Centering and normalization

Let $H=I_n-\frac{1}{n}\mathbf{1}\mathbf{1}^{\mathsf T}$ and define $X_c=HX$, $Y_c=HY$. This subtracts each neuron's mean across examples. It removes constant activation offsets so CKA and Procrustes compare variation across stimuli. The Frobenius norm is $\|A\|_F=\sqrt{\sum_{ij}A_{ij}^2}$.

We do **not** divide individual columns by their standard deviations. That would change the geometry by reweighting neurons and amplify nearly constant units. Global normalization removes overall scale while preserving relative neuron contributions. Post-ReLU activations include zeros for inactive units; this sparsity is part of the representation being analyzed.

### Orthogonal Procrustes

Normalize $\widetilde X=X_c/\|X_c\|_F$ and $\widetilde Y=Y_c/\|Y_c\|_F$. Seek an orthogonal matrix $Q$ that aligns neuron coordinates:

$$\min_{Q^{\mathsf T}Q=I}\|\widetilde XQ-\widetilde Y\|_F^2.$$

If $\widetilde X^{\mathsf T}\widetilde Y=U\Sigma V^{\mathsf T}$, the optimal alignment is $Q=UV^{\mathsf T}$. The code only needs the singular values:

$$s_P=\sum_k\Sigma_{kk}=\frac{\|X_c^{\mathsf T}Y_c\|_*}{\|X_c\|_F\|Y_c\|_F},\qquad 0\leq s_P\leq1.$$

Here $\|\cdot\|_*$ is the sum of singular values. Higher means a better alignment. Orthogonality permits rotations and reflections of neuron coordinates while preventing arbitrary shears. The normalized orthogonal residual is $2-2s_P$.

The reported **Procrustes disparity** is instead $1-s_P^2$: after normalizing both matrices, also optimize a scalar $a$ in $\min_{a,Q}\|\widetilde X-a\widetilde YQ\|_F^2$. The optimum is $a=s_P$, producing $1-s_P^2$. This is the convention used by SciPy's spatial Procrustes calculation; it is distinct from the orthogonal-only residual. Lower disparity means closer alignment.

### Linear centered kernel alignment (CKA)

$$\operatorname{CKA}(X,Y)=\frac{\|X_c^{\mathsf T}Y_c\|_F^2}{\|X_c^{\mathsf T}X_c\|_F\,\|Y_c^{\mathsf T}Y_c\|_F}.$$

Equivalently, this is the Frobenius cosine similarity of centered sample Gram matrices $K=X_cX_c^{\mathsf T}$ and $L=Y_cY_c^{\mathsf T}$. The feature-space formula above avoids allocating those $n\times n$ matrices. Linear kernels compare geometry without introducing a kernel bandwidth. CKA is invariant to translation (through centering), orthogonal coordinate changes, and nonzero uniform scaling; it is not generally invariant to arbitrary invertible linear transformations.

The score is in $[0,1]$, with identical nonconstant representations giving 1. This is the biased empirical estimator, without a finite-sample correction. Scores depend on the sample and are descriptive rather than calibrated significance measures.

### Representational similarity analysis (RSA)

For each layer, construct Euclidean distances between stimulus rows:

$$D^X_{ij}=\|X_i-X_j\|_2,\qquad D^Y_{ij}=\|Y_i-Y_j\|_2.$$

Euclidean distance measures absolute differences across neuron activations. It accommodates individual zero activation vectors, unlike correlation distance, which may be undefined for constant vectors. Translation does not change pairwise distances, so explicit centering is unnecessary here.

Extract only $i<j$: the diagonal is trivially zero and the lower triangle duplicates the upper triangle. Let $u=\operatorname{vec}_{i<j}(D^X)$ and $v=\operatorname{vec}_{i<j}(D^Y)$. Compute

$$\rho_{\mathrm{RSA}}=\operatorname{corr}(\operatorname{rank}(u),\operatorname{rank}(v)).$$

Spearman correlation uses average ranks for ties. It measures agreement in **distance ordering**, rather than exact magnitudes. Its range is $[-1,1]$: positive means similar orderings, zero means no rank correlation, and negative means reversed orderings. Uniform positive scale and orthogonal changes preserve these rankings. This differs from CKA and Procrustes, which retain more magnitude information.

### Undefined cases and numerical safeguards

Constant centered representations have zero norm, making CKA and Procrustes undefined. RSA is undefined if either distance vector is constant. Return NaN instead of declaring those cases perfect matches. Require at least three matched samples, giving multiple distinct pairs for RSA. Clip computed CKA and Procrustes scores to $[0,1]$ only to remove floating-point overshoot. No clipping is applied to the mathematical inputs.


In [ ]:
def centered(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError("Expected a finite samples-by-neurons matrix")
    return x - x.mean(axis=0, keepdims=True)


def procrustes_similarity(x, y):
    x, y = centered(x), centered(y)
    denom = np.linalg.norm(x) * np.linalg.norm(y)
    if denom == 0:
        return np.nan
    return float(np.clip(np.linalg.svd(x.T @ y, compute_uv=False).sum() / denom, 0, 1))


def linear_cka(x, y):
    x, y = centered(x), centered(y)
    denom = np.linalg.norm(x.T @ x, "fro") * np.linalg.norm(y.T @ y, "fro")
    if denom == 0:
        return np.nan
    return float(np.clip(np.linalg.norm(x.T @ y, "fro") ** 2 / denom, 0, 1))


def rsa_similarity(x, y):
    dx, dy = pdist(x, metric="euclidean"), pdist(y, metric="euclidean")
    if np.ptp(dx) == 0 or np.ptp(dy) == 0:
        return np.nan
    return float(spearmanr(dx, dy).statistic)


def compare_layers(x, y):
    if x.shape[0] != y.shape[0] or x.shape[0] < 3:
        raise ValueError("Need at least three matched examples")
    proc = procrustes_similarity(x, y)
    return {
        "Procrustes similarity": proc,
        "Procrustes disparity": 1 - proc ** 2,
        "Linear CKA": linear_cka(x, y),
        "RSA (Spearman)": rsa_similarity(x, y),
    }


## Why these sanity checks?

An identical representation should score 1 under every similarity metric. The synthetic transformation $Y=3XQ+b$ combines a positive uniform scale, an orthogonal matrix from QR decomposition, and a translation. All three metrics should also give 1 for that transformation, testing their intended invariances. A constant matrix should give undefined CKA. These checks test formulas, not checkpoint performance; they do not validate the model's learned tasks.


In [ ]:
# Check identity and invariance to translation, uniform scale, and rotation.
rng = np.random.default_rng(SEED)
x_check = rng.normal(size=(40, 16))
q, _ = np.linalg.qr(rng.normal(size=(16, 16)))
y_check = 3 * x_check @ q + rng.normal(size=16)
for metric in [procrustes_similarity, linear_cka, rsa_similarity]:
    assert np.isclose(metric(x_check, x_check), 1)
    assert np.isclose(metric(x_check, y_check), 1)
assert np.isnan(linear_cka(np.ones((10, 16)), np.ones((10, 16))))
print("Metric sanity checks passed")


In [ ]:
rows = []
for branch, (first, last) in comparisons.items():
    rows.append({
        "Branch": branch,
        "First layer": first,
        "Last layer": last,
        **compare_layers(activations[first], activations[last]),
    })
results = pd.DataFrame(rows).set_index("Branch")
display(results.round(4))

scores = ["Procrustes similarity", "Linear CKA", "RSA (Spearman)"]
score_values = results[scores].to_numpy()
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(np.ma.masked_invalid(score_values), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(scores)), scores)
ax.set_yticks(range(len(results)), results.index)
for i in range(len(results)):
    for j in range(len(scores)):
        value = score_values[i, j]
        ax.text(j, i, f"{value:.3f}" if np.isfinite(value) else "NaN",
                ha="center", va="center",
                color="white" if np.isfinite(value) and abs(value) > 0.6 else "black")
ax.set_title("Deep checkpoint: first versus last hidden layer")
fig.colorbar(im, ax=ax, label="Similarity (zero-centered scale)")
plt.tight_layout()
plt.show()


## Diverging colormaps and representational distances

The similarity heatmap uses `RdBu_r` on the fixed interval $[-1,1]$: blue is negative, white is zero, and red is positive. Only RSA can be negative; CKA and Procrustes occupy the nonnegative half. Disparity stays in the table because its direction is reversed (lower is better). Sharing colors does not make the three metrics numerically interchangeable. NaN cells are masked and labeled explicitly.

For raw distance matrices, use the same diverging palette with limits $[0,D_{\max}]$ shared by the first/last panels of each branch. **Distances cannot be negative:** blue means small distance, red means large distance, and white is the visual midpoint $D_{\max}/2$, not zero or a statistical baseline. Shared limits allow magnitude comparisons within a branch, but separate branch scales prevent interpreting colors across branches as equal distances.

Show the first 60 cached examples for readability; metric calculations still use all collected examples. These matrices are symmetric with zero diagonals because they contain Euclidean distances.

A second plot gives the diverging palette a signed interpretation. Normalize each displayed RDM by its own RMS off-diagonal distance:

$$s_X=\sqrt{\frac{1}{m}\sum_{i<j}(D^X_{ij})^2},\quad \widehat D^X=D^X/s_X,\quad \Delta D=\widehat D^Y-\widehat D^X,$$

where $m=k(k-1)/2$ for the displayed $k$ examples. RMS normalization removes global distance scale so the difference emphasizes relative geometry. Red means a pair became relatively farther apart in the last layer; blue means relatively closer; white means unchanged normalized distance. Use symmetric limits $[-M,M]$, with one $M=\max|\Delta D|$ across branches. These normalized differences are a visual aid, not the definition of RSA, and exclude any alignment or fitted coordinate transformation. If an RDM has zero RMS, its normalized difference is undefined and masked.


In [ ]:
N_DISPLAY = min(60, collected)
fig, axes = plt.subplots(len(comparisons), 2, figsize=(10, 16))
for row, (branch, pair) in enumerate(comparisons.items()):
    rdms = [squareform(pdist(activations[name][:N_DISPLAY])) for name in pair]
    vmax = max(rdm.max() for rdm in rdms)
    for col, (name, rdm) in enumerate(zip(pair, rdms)):
        ax = axes[row, col]
        im = ax.imshow(rdm, cmap="RdBu_r", vmin=0, vmax=vmax)
        ax.set_title(name)
        ax.set_xlabel("Test example")
        ax.set_ylabel("Test example")
    fig.colorbar(im, ax=list(axes[row]), label="Euclidean distance", shrink=0.8)
fig.suptitle("Matched representational distance matrices", y=0.995)
plt.show()

# Signed differences after removing each displayed RDM's global scale.
differences = []
for branch, pair in comparisons.items():
    distances = [pdist(activations[name][:N_DISPLAY]) for name in pair]
    scales = [np.sqrt(np.mean(d ** 2)) for d in distances]
    if any(scale == 0 for scale in scales):
        differences.append(np.full((N_DISPLAY, N_DISPLAY), np.nan))
    else:
        differences.append(squareform(distances[1] / scales[1] - distances[0] / scales[0]))
finite_values = np.concatenate([d[np.isfinite(d)] for d in differences])
limit = max(np.abs(finite_values).max(), np.finfo(float).eps) if finite_values.size else 1.0
fig, axes = plt.subplots(1, len(comparisons), figsize=(16, 4), constrained_layout=True)
for ax, branch, difference in zip(axes, comparisons, differences):
    im = ax.imshow(np.ma.masked_invalid(difference), cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_title(branch)
    ax.set_xlabel("Test example")
    ax.set_ylabel("Test example")
fig.colorbar(im, ax=list(axes), label="Last - first (RMS-normalized distance)", shrink=0.8)
fig.suptitle("Changes in relative pairwise distances")
plt.show()


## Reading the results

Higher similarity means that the first and last hidden layers preserve more of the same geometry under the chosen metric. Lower similarity indicates a larger transformation; it does not by itself indicate better learning. High RSA can coexist with lower CKA because RSA compares distance ranks rather than their magnitudes.

Treat these as descriptive results for one checkpoint and one test sample. The four comparisons cover different paths: the shared comparison spans the backbone, while each task comparison spans its branch. Depth, ReLU sparsity, the narrow 16-neuron representation, and the original training objective can all affect the scores. Establishing task selectivity would require additional analyses using digit, parity, and rotation labels. No statistical independence or significance is assumed for the pairwise distances.
